In [1]:
import os
from pathlib import Path

import numpy as np
import yaml


In [2]:
def repo_root():
    env = os.environ.get("REALTIME_ALIGNMENT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "forAkshay":
        return cwd.parent
    if (cwd / "forAkshay").is_dir():
        return cwd
    return cwd



ROOT = repo_root()
ONNX_NARROW = ROOT / "onnx_no-residual" / "onnx_files_narrow"
SIMPLE_DIR = ONNX_NARROW / "simple_full"
FULL_ONNX = ONNX_NARROW / "mlp_full.onnx"
CONFIG_PATH = ROOT / "onnx_no-residual" / "checkpoints" / "config_small.yaml"

In [3]:
print("ROOT =", ROOT)
print("SIMPLE_DIR =", SIMPLE_DIR)
print("FULL_ONNX =", FULL_ONNX)

ROOT = /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment
SIMPLE_DIR = /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/onnx_no-residual/onnx_files_narrow/simple_full
FULL_ONNX = /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/onnx_no-residual/onnx_files_narrow/mlp_full.onnx


In [4]:
with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

m = cfg["model"]
subset_size = m["subset_config"][0][0]
num_solvers = len(m["subset_config"])
num_particles = cfg["data"]["num_particles"]
in_features = m["in_features"]
out_features = m["out_features"]

print(f"num_particles={num_particles}, in_features={in_features}, out_features={out_features}," f"subset_size={subset_size}, num_solvers={num_solvers}")

num_particles=50, in_features=6, out_features=27,subset_size=2, num_solvers=3


In [5]:
cfg

{'checkpointing': {'checkpoint_path': '/home/yhuang2/PROJs/RealTimeAlignment/train/mlp_no-residual/checkpoints_small',
  'resume': True,
  'save_frequency': 20},
 'data': {'mode': 'raw', 'num_particles': 50, 'rounded': False},
 'model': {'embedding_features': [192, 192],
  'in_features': 6,
  'out_features': 27,
  'norm': None,
  'activ': {'name': 'leakyrelu', 'negative_slope': 0.1},
  'subset_config': [[2, 192, 192, 192, 192],
   [2, 192, 192, 192, 192],
   [2, 192, 192, 192, 192]]},
 'train': {'batch_size': 64,
  'learning_rate': 0.0001,
  'num_epochs': 200,
  'num_warmup_epochs': 50,
  'sched_gamma': 0.95,
  'sched_steps': 20}}

In [6]:
import onnxruntime as ort


def run_onnx_full(session: ort.InferenceSession, x: np.ndarray):
    """Full MLP: x shape (batch, 50, 6) -> (batch, 27)."""
    name = session.get_inputs()[0].name
    return session.run(None, {name: np.asarray(x, dtype=np.float32)})[0]


def run_vec(session: ort.InferenceSession, v: np.ndarray):
    """Single rank-1 forward for simple_* ONNX (one particle)."""
    name = session.get_inputs()[0].name
    v = np.asarray(v, dtype=np.float32).reshape(-1)
    return session.run(None, {name: v})[0]


def apply_vec_session_to_cloud(session: ort.InferenceSession, cloud: np.ndarray):
    """
    cloud: (1, num_particles, C_in) — run the 1D graph on each particle row.
    Returns (1, num_particles, C_out).
    """
    b, p, _ = cloud.shape
    assert b == 1, cloud.shape
    rows = [run_vec(session, cloud[0, j]) for j in range(p)]
    return np.stack(rows, axis=0)[np.newaxis, :, :]


def assemble_np(a: np.ndarray, subset_size: int):
    return np.concatenate([np.roll(a, shift=i, axis=1) for i in range(subset_size)], axis=-1)




def summarize(tag: str, a: np.ndarray):
    a = np.asarray(a)
    print(f"\n[{tag}]")
    print(f"  shape: {a.shape}  dtype: {a.dtype}")
    print(f"  min: {a.min():.6g}  max: {a.max():.6g}  mean: {a.mean():.6g}")

In [7]:
EMBED_ONNX = SIMPLE_DIR / "simple_submodule_embed.onnx"
SOLVER0_ONNX = SIMPLE_DIR / "simple_submodule_solvers-0.onnx"
SOLVER1_ONNX = SIMPLE_DIR / "simple_submodule_solvers-1.onnx"
SOLVER2_ONNX = SIMPLE_DIR / "simple_submodule_solvers-2.onnx"
OUTPUT_ONNX = SIMPLE_DIR / "simple_submodule_output.onnx"
for path in (EMBED_ONNX, SOLVER0_ONNX):
    if not path.is_file():
        raise FileNotFoundError(path)

sess_embed = ort.InferenceSession(str(EMBED_ONNX), providers=["CPUExecutionProvider"])
sess_solver0 = ort.InferenceSession(str(SOLVER0_ONNX), providers=["CPUExecutionProvider"])
sess_solver1 = ort.InferenceSession(str(SOLVER1_ONNX), providers=["CPUExecutionProvider"])
sess_solver2 = ort.InferenceSession(str(SOLVER2_ONNX), providers=["CPUExecutionProvider"])
sess_output = ort.InferenceSession(str(OUTPUT_ONNX), providers=["CPUExecutionProvider"])

for label, path, s in (("embed", EMBED_ONNX, sess_embed), ("solver-0", SOLVER0_ONNX, sess_solver0), ("solver-1", SOLVER1_ONNX, sess_solver1), ("solver-2", SOLVER2_ONNX, sess_solver2),("output", OUTPUT_ONNX, sess_output)):
    inp, out = s.get_inputs()[0], s.get_outputs()[0]
    print(f"Loaded {label}: {path.name}")
    print(f"  input:  name={inp.name!r} shape={inp.shape} type={inp.type}")
    print(f"  output: name={out.name!r} shape={out.shape} type={out.type}")
print("Each simple graph is rank-1 per particle; stack to (1, 50, C). After embed, assemble_np then solver-0.")

Loaded embed: simple_submodule_embed.onnx
  input:  name='input' shape=[6] type=tensor(float)
  output: name='output' shape=[] type=tensor(float)
Loaded solver-0: simple_submodule_solvers-0.onnx
  input:  name='input' shape=[256] type=tensor(float)
  output: name='output' shape=[] type=tensor(float)
Loaded solver-1: simple_submodule_solvers-1.onnx
  input:  name='input' shape=[256] type=tensor(float)
  output: name='output' shape=[] type=tensor(float)
Loaded solver-2: simple_submodule_solvers-2.onnx
  input:  name='input' shape=[256] type=tensor(float)
  output: name='output' shape=[] type=tensor(float)
Loaded output: simple_submodule_output.onnx
  input:  name='input' shape=[128] type=tensor(float)
  output: name='output' shape=[] type=tensor(float)
Each simple graph is rank-1 per particle; stack to (1, 50, C). After embed, assemble_np then solver-0.


2026-04-13 14:17:25.197902036 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.
2026-04-13 14:17:25.225850899 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.
2026-04-13 14:17:25.255112539 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.
2026-04-13 14:17:25.280862876 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.
2026-04-13 14:17:25.298724400 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{27} target:{-1,50,27}. Falling back to lenient merge.


In [8]:
# Use inputs generated by export_io_txt.py (inputs_10.txt)
inputs_txt = ROOT / "forAkshay" / "inputs_10.txt"
if not inputs_txt.is_file():
    raise FileNotFoundError(inputs_txt)

# File has one header line starting with '#'; each row has 300 floats (=50*6)
rows = np.loadtxt(inputs_txt, comments="#", dtype=np.float32)
if rows.ndim == 1:
    rows = rows[np.newaxis, :]

assert rows.shape[1] == num_particles * in_features, rows.shape
print(f"Loaded {rows.shape[0]} rows from {inputs_txt}")

sample_idx = 0  # change to 0..9
x1 = rows[sample_idx : sample_idx + 1].reshape(1, num_particles, in_features)
summarize(f"input sample {sample_idx}", x1)

Loaded 10 rows from /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/inputs_10.txt

[input sample 0]
  shape: (1, 50, 6)  dtype: float32
  min: -1.13444  max: 1.34846  mean: -0.00194199


# Doing things step wise!!

In [9]:
h = apply_vec_session_to_cloud(sess_embed, x1)
summarize("simple embed output", h)

h_assembled = assemble_np(h, subset_size)
summarize("assembled (embed rolls+concat)", h_assembled)

np.set_printoptions(precision=6, suppress=False, linewidth=160)
print("\nAssembled: particle 0 (all dims):")
print(h_assembled[0, 0, :])
print("\nAssembled preview: first 3 particles, first 8 dims")
print(h_assembled[0, :3, :8])


[simple embed output]
  shape: (1, 50, 128)  dtype: float32
  min: -0.0921552  max: 0.961171  mean: 0.103591

[assembled (embed rolls+concat)]
  shape: (1, 50, 256)  dtype: float32
  min: -0.0921552  max: 0.961171  mean: 0.103591

Assembled: particle 0 (all dims):
[ 1.389675e-02  1.202718e-01  5.349925e-01  8.418708e-02  4.225016e-02 -2.555402e-02 -2.893185e-02 -2.002839e-03 -3.847305e-02  4.990914e-01 -1.807672e-02
  4.177611e-02 -1.183949e-02  4.219648e-01 -1.093767e-03 -2.131952e-02 -5.715196e-03  3.721696e-01  4.340074e-01 -4.235378e-02  1.869244e-02  2.549295e-01
  1.188918e-01 -1.716243e-02  7.524696e-02  1.644705e-02  3.070995e-01 -1.592429e-03 -4.457122e-02  2.965132e-01  2.964435e-01  1.506526e-01 -2.960910e-02
  2.705601e-01 -9.010658e-05  4.077647e-02 -6.947446e-02 -1.658728e-02  2.227713e-01  2.496511e-02  4.457185e-02 -1.482618e-02  2.232197e-01  8.106627e-02
  2.301659e-01  9.903979e-02  2.244787e-02 -1.812194e-02  3.546737e-01  3.008924e-01  2.011351e-01 -1.882444e-02  

In [10]:
h_solv0 = apply_vec_session_to_cloud(sess_solver0, h_assembled)
summarize("simple solver-0 output", h_solv0)

print("\nSolver-0: particle 0 (all dims):")
print(h_solv0[0, 0, :])


[simple solver-0 output]
  shape: (1, 50, 128)  dtype: float32
  min: -0.0755976  max: 0.845579  mean: 0.144197

Solver-0: particle 0 (all dims):
[ 2.304585e-01  2.823796e-01  5.345124e-01  6.742717e-02 -4.667244e-02  1.638968e-01  2.505605e-02 -7.675934e-03 -5.376601e-03 -2.083504e-02 -4.329193e-03
  5.454445e-01  2.619973e-01 -2.475202e-02  1.465536e-01 -6.397981e-03 -2.015129e-02 -3.886217e-03  1.379802e-01  4.640816e-01  6.090004e-02  3.242669e-01
 -1.680127e-02  1.162041e-01  1.732870e-01 -1.682008e-02 -7.929548e-03  2.370021e-01  1.354148e-01  2.104024e-01  2.330004e-01  3.110631e-01  1.427854e-01
 -2.574586e-02  6.178372e-03  3.259627e-01 -1.014375e-02  6.475650e-01 -8.949513e-03 -2.991626e-03  4.578117e-01  5.405202e-01 -9.376159e-03  9.646085e-02
 -1.403870e-03 -2.452354e-02  1.729688e-01  2.224551e-01  2.059475e-01  7.677773e-02  4.418049e-01  3.900407e-02  2.393637e-01  1.418039e-01 -5.181506e-03
  2.426244e-01 -2.797676e-02  1.493821e-01 -1.126159e-02 -5.709912e-03  6.2902

In [11]:
h_assembled_1 = assemble_np(h_solv0, subset_size)
summarize("assembled for solver-1", h_assembled_1)

h_solv1 = apply_vec_session_to_cloud(sess_solver1, h_assembled_1)
summarize("simple solver-1 output", h_solv1)

print("\nSolver-1: particle 0 (all dims):")
print(h_solv1[0, 0, :])


[assembled for solver-1]
  shape: (1, 50, 256)  dtype: float32
  min: -0.0755976  max: 0.845579  mean: 0.144197

[simple solver-1 output]
  shape: (1, 50, 128)  dtype: float32
  min: -0.1344  max: 1.29732  mean: 0.163335

Solver-1: particle 0 (all dims):
[-2.648034e-02  2.174676e-01  5.119814e-01 -1.656125e-02  6.761707e-02 -2.035803e-02  4.068579e-01 -8.954172e-02  1.722503e-01 -2.544695e-02  3.605750e-01
  1.401922e-02 -3.370795e-02  2.805154e-01  6.693679e-02  4.072479e-01 -9.557997e-03  3.725380e-01 -5.308817e-03  2.406628e-01 -1.866160e-02  4.112545e-01
  4.439946e-02  3.013026e-01 -2.105338e-02 -2.541697e-02  1.980417e-01 -3.050881e-02  6.156502e-02  5.184469e-01  4.335447e-01  1.169377e-01  5.639294e-02
 -9.622092e-03 -2.187853e-02  2.831562e-01  2.158288e-01  4.459701e-01  3.542663e-01 -1.846547e-02 -3.833780e-03 -9.137438e-03  3.604270e-01  7.115424e-02
 -1.745678e-02  6.785663e-02 -1.274991e-02  2.655720e-01 -8.503998e-03  1.119137e-01  7.283016e-02  6.473176e-01  1.368929e-

In [12]:
h_assembled_2 = assemble_np(h_solv1, subset_size)
summarize("assembled for solver-2", h_assembled_2)

h_solv2 = apply_vec_session_to_cloud(sess_solver2, h_assembled_2)
summarize("simple solver-2 output", h_solv2)

print("\nSolver-2: particle 0 (all dims):")
print(h_solv2[0, 0, :])


[assembled for solver-2]
  shape: (1, 50, 256)  dtype: float32
  min: -0.1344  max: 1.29732  mean: 0.163335

[simple solver-2 output]
  shape: (1, 50, 128)  dtype: float32
  min: -0.0654964  max: 0.489804  mean: 0.0294873

Solver-2: particle 0 (all dims):
[ 7.969898e-02  1.157269e-01  8.008480e-02 -7.562800e-03  1.347977e-01  2.266190e-01  1.694949e-01 -4.459966e-03 -4.446887e-03 -2.049425e-02 -4.544265e-03
 -1.901296e-02 -2.335016e-02  1.796900e-02 -8.370507e-03 -4.551754e-03 -6.851987e-03  6.340248e-02 -1.162361e-03  1.633034e-02 -5.125333e-03 -1.089095e-02
 -8.378404e-03 -9.807645e-03  4.209851e-02 -7.416143e-03 -2.225028e-03 -9.314120e-03 -2.117633e-02  4.308111e-02 -2.308156e-03 -7.140425e-03  1.628388e-02
  1.716307e-02 -1.256327e-04  3.725632e-02  1.194459e-01  2.621097e-02 -4.238603e-03 -3.331652e-03 -3.775771e-03 -6.564897e-03 -8.629689e-03 -9.099970e-03
 -8.496020e-03  1.482146e-01  5.643043e-02  5.572770e-02 -7.378802e-03  5.863978e-02  1.370668e-01  4.348138e-02  1.826506e

In [13]:
h_output = apply_vec_session_to_cloud(sess_output, h_solv2)
summarize("simple output", h_output)

print("\nOutput: particle 0 (all dims):")
print(h_output[0, 0, :])



[simple output]
  shape: (1, 50, 27)  dtype: float32
  min: -0.332097  max: 0.378391  mean: -0.00558264

Output: particle 0 (all dims):
[-0.018267 -0.021173  0.008397  0.00402   0.0235    0.019226  0.022721 -0.031957 -0.003569 -0.018438 -0.008777 -0.006612  0.007359 -0.112468 -0.023977
 -0.015451 -0.046968 -0.008132 -0.012321 -0.043871 -0.038103 -0.002121 -0.099424 -0.012719 -0.023519  0.021028 -0.00119 ]


In [14]:
# Compare simple stepwise output vs full model_fp32 output on the same input sample
sess_full = ort.InferenceSession(str(FULL_ONNX), providers=["CPUExecutionProvider"])
y_full = run_onnx_full(sess_full, x1)

summarize("full model_fp32 output", y_full)

# simple output is (1, 50, 27) while full model output is (1, 27)
y_simple_reduced = h_output.mean(axis=1)  # reduce particle axis for direct compare
summarize("simple output reduced (mean over 50 particles)", y_simple_reduced)

diff = y_simple_reduced - y_full
abs_diff = np.abs(diff)

np.set_printoptions(precision=6, suppress=False, linewidth=160)
print("\nfull model_fp32 output (sample 0, all 27 dims):")
print(y_full[0, :])
print("\nsimple reduced output (sample 0, all 27 dims):")
print(y_simple_reduced[0, :])
print("\ndiff (simple_reduced - full, sample 0):")
print(diff[0, :])
print(f"\nmax|diff|={abs_diff.max():.6g}, mean|diff|={abs_diff.mean():.6g}, rmse={np.sqrt(np.mean(diff**2)):.6g}")


[full model_fp32 output]
  shape: (1, 27)  dtype: float32
  min: -0.0421008  max: 0.0305853  mean: -0.00558264

[simple output reduced (mean over 50 particles)]
  shape: (1, 27)  dtype: float32
  min: -0.0421007  max: 0.0305853  mean: -0.00558264

full model_fp32 output (sample 0, all 27 dims):
[ 0.006879 -0.038808  0.008011  0.00038   0.007102  0.002953  0.003943  0.003111 -0.000906 -0.008843  0.012984 -0.002185  0.00522  -0.026135 -0.018964
 -0.023803  0.030585 -0.008492 -0.006166 -0.001007 -0.016249  0.002377 -0.010913 -0.039594 -0.042101  0.011539 -0.001649]

simple reduced output (sample 0, all 27 dims):
[ 0.006879 -0.038808  0.008011  0.00038   0.007102  0.002953  0.003943  0.003111 -0.000906 -0.008843  0.012984 -0.002185  0.00522  -0.026135 -0.018964
 -0.023803  0.030585 -0.008492 -0.006166 -0.001007 -0.016249  0.002377 -0.010913 -0.039594 -0.042101  0.011539 -0.001649]

diff (simple_reduced - full, sample 0):
[ 6.519258e-09  1.117587e-08 -9.313226e-10 -2.328306e-09  5.587935e-

In [ ]:
# Upgraded dump: params + runtime activations in Akshay-compatible format
# Run after sessions/x1 are defined (cells 6-7)

from pathlib import Path
from typing import Optional
import sys



sys.path = [
    p for p in sys.path
    if not ("/.local/lib/" in p and "/site-packages" in p)
]
for mod in list(sys.modules):
    if mod.startswith("google") or mod.startswith("protobuf"):
        sys.modules.pop(mod, None)



try:
    import onnx
    from onnx import numpy_helper
except Exception as e:
    raise RuntimeError(
        "Failed to import onnx in this kernel due to package conflict. "
        "Try running notebook with PYTHONNOUSERSITE=1 or align protobuf versions. "
        f"Original error: {e}"
    ) from e


def _sanitize(name: str) -> str:
    for ch in ['/', '\\', ':', '*', '?', '"', '<', '>', '|', ' ']:
        name = name.replace(ch, '_')
    return name


def _dump_txt(path: Path, arr: np.ndarray, fmt: str = "%.8g") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    flat = np.asarray(arr).reshape(-1)
    with path.open("w", encoding="utf-8") as f:
        for v in flat:
            f.write((fmt % float(v)) + "\n")

def _build_maps(model):
    g = model.graph
    init_map = {init.name: numpy_helper.to_array(init) for init in g.initializer}
    consumers = {}
    for n in g.node:
        for i in n.input:
            consumers.setdefault(i, []).append(n)
    return init_map, consumers


def _list_dense_and_lrelu(model):
    g = model.graph
    init_map, consumers = _build_maps(model)

    denses = []
    for n in g.node:
        if n.op_type == "Gemm":
            w = n.input[1] if len(n.input) > 1 else None
            b = n.input[2] if len(n.input) > 2 else None
            denses.append({"out": n.output[0], "W": init_map.get(w), "B": init_map.get(b)})
        elif n.op_type == "MatMul":
            mm_out = n.output[0]
            w = n.input[1] if len(n.input) > 1 else None
            b_arr = None
            out = mm_out
            for c in consumers.get(mm_out, []):
                if c.op_type == "Add":
                    other = c.input[1] if c.input[0] == mm_out else c.input[0]
                    b_arr = init_map.get(other)
                    out = c.output[0]
                    break
            denses.append({"out": out, "W": init_map.get(w), "B": b_arr})

    lrs = []
    for n in g.node:
        if n.op_type == "LeakyRelu":
            alpha = 0.01
            for a in n.attribute:
                if a.name == "alpha":
                    alpha = float(a.f)
            lrs.append({"out": n.output[0], "alpha": alpha})

    return denses, lrs


def _instrument_outputs(model, names):
    m = onnx.ModelProto()
    m.CopyFrom(model)
    existing = {o.name for o in m.graph.output}
    for name in names:
        if name not in existing:
            vi = onnx.helper.make_tensor_value_info(name, onnx.TensorProto.FLOAT, None)
            m.graph.output.append(vi)
    return m


def dump_stage_with_activations(onnx_path: Path, tag: str, x_in: np.ndarray, out_root: Path):
    out_dir = out_root / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    model = onnx.load(str(onnx_path))
    denses, lrs = _list_dense_and_lrelu(model)

    names_to_fetch = [d["out"] for d in denses] + [lr["out"] for lr in lrs]
    orig_out_names = [o.name for o in model.graph.output]

    inst = _instrument_outputs(model, names_to_fetch)
    tmp = ROOT / "forAkshay" / f"tmp_{tag.lower()}_inst.onnx"
    onnx.save(inst, str(tmp))

    sess = ort.InferenceSession(str(tmp), providers=["CPUExecutionProvider"])
    in_name = sess.get_inputs()[0].name

    # Dump exact stage input (flattened text as in Akshay dumps)
    _dump_txt(out_dir / "input.txt", np.asarray(x_in), fmt="%.8g")

    fetches = names_to_fetch + orig_out_names

    # Handle rank-1 stage inputs (simple_submodule_*): run one particle at a time and stack.
    in_rank = len(sess.get_inputs()[0].shape)
    if in_rank == 1 and np.asarray(x_in).ndim == 3:
        b, p, _ = x_in.shape
        assert b == 1, f"Expected batch=1 for rank-1 stage dump, got {x_in.shape}"

        per_fetch = {name: [] for name in fetches}
        for j in range(p):
            v = np.asarray(x_in[0, j], dtype=np.float32).reshape(-1)
            vals = sess.run(fetches, {in_name: v})
            for name, arr in zip(fetches, vals):
                per_fetch[name].append(np.asarray(arr))

        # Stack each fetched tensor back to (1, P, C)
        stacked = {}
        for name in fetches:
            a = np.stack(per_fetch[name], axis=0)  # (P, C)
            stacked[name] = a[np.newaxis, ...]     # (1, P, C)

        inter_map = {name: stacked[name] for name in names_to_fetch}
        out_map = {name: stacked[name] for name in orig_out_names}
        stage_output = out_map[orig_out_names[0]]
    else:
        # Normal dense/full stage path
        vals = sess.run(fetches, {in_name: np.asarray(x_in, dtype=np.float32)})
        inter_arrs = vals[:len(names_to_fetch)]
        graph_arrs = vals[len(names_to_fetch):]
        inter_map = dict(zip(names_to_fetch, inter_arrs))
        out_map = dict(zip(orig_out_names, graph_arrs))
        stage_output = np.asarray(out_map[orig_out_names[0]])

    # stage_output and canonical model_output
    _dump_txt(out_dir / "stage_output.txt", np.asarray(stage_output), fmt="%.8g")

    with (out_dir / "graph_outputs.txt").open("w", encoding="utf-8") as f:
        for n in orig_out_names:
            f.write(n + "\n")
    _dump_txt(out_dir / "model_output.txt", np.asarray(out_map[orig_out_names[0]]), fmt="%.8g")

    # Dense dumps (weights/bias/output)
    for i, d in enumerate(denses):
        if d["W"] is not None:
            _dump_txt(out_dir / f"dense_{i}_weights.txt", np.asarray(d["W"]), fmt="%.8g")
        if d["B"] is not None:
            _dump_txt(out_dir / f"dense_{i}_bias.txt", np.asarray(d["B"]), fmt="%.8g")
        _dump_txt(out_dir / f"dense_{i}_output.txt", np.asarray(inter_map[d["out"]]), fmt="%.8g")

    # LeakyReLU dumps (output + alpha)
    for i, lr in enumerate(lrs):
        _dump_txt(out_dir / f"leakyrelu_{i}_output.txt", np.asarray(inter_map[lr["out"]]), fmt="%.8g")
        _dump_txt(out_dir / f"leakyrelu_{i}_alpha.txt", np.asarray([lr["alpha"]]), fmt="%.8g")

    print(f"[{tag}] OK: dumped to {out_dir}")
    return np.asarray(stage_output)


def dump_all_submodels_with_activations(sample: np.ndarray, output_root: Optional[Path] = None):
    if output_root is None:
        output_root = ROOT / "forAkshay" / "onnx_txt"
    output_root.mkdir(parents=True, exist_ok=True)

    x = np.asarray(sample, dtype=np.float32)
    x = dump_stage_with_activations(EMBED_ONNX, "EMBED", x, output_root)

    x = assemble_np(x, subset_size)
    x = dump_stage_with_activations(SOLVER0_ONNX, "SOLVER-0", x, output_root)

    x = assemble_np(x, subset_size)
    x = dump_stage_with_activations(SOLVER1_ONNX, "SOLVER-1", x, output_root)

    x = assemble_np(x, subset_size)
    x = dump_stage_with_activations(SOLVER2_ONNX, "SOLVER-2", x, output_root)

    x = dump_stage_with_activations(OUTPUT_ONNX, "OUTPUT", x, output_root)

    print(f"Chain complete. Dumps are under: {output_root}")
    return x



In [30]:
dump_all_submodels_with_activations(x1)

2026-04-13 14:32:23.367680140 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.
2026-04-13 14:32:23.488647147 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.


[EMBED] OK: dumped to /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/onnx_txt/EMBED
[SOLVER-0] OK: dumped to /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/onnx_txt/SOLVER-0


2026-04-13 14:32:23.742827457 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.


[SOLVER-1] OK: dumped to /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/onnx_txt/SOLVER-1


2026-04-13 14:32:24.163968143 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{128} target:{-1,50,128}. Falling back to lenient merge.


[SOLVER-2] OK: dumped to /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/onnx_txt/SOLVER-2
[OUTPUT] OK: dumped to /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/onnx_txt/OUTPUT
Chain complete. Dumps are under: /gpfs/mnt/gpfs01/usfcc/pusharma/Haider_LDRD/RealTimeAlignment/forAkshay/onnx_txt


2026-04-13 14:32:24.818628848 [W:onnxruntime:, graph.cc:109 MergeShapeInfo] Error merging shape info for output. 'output' source:{27} target:{-1,50,27}. Falling back to lenient merge.


array([[[-0.006434, -0.035434,  0.031035, ..., -0.058254, -0.015166, -0.001011],
        [-0.006434, -0.035434,  0.031035, ..., -0.058254, -0.015166, -0.001011],
        [-0.006434, -0.035434,  0.031035, ..., -0.058254, -0.015166, -0.001011],
        ...,
        [-0.002159, -0.038597,  0.047667, ..., -0.07307 , -0.01004 , -0.002751],
        [-0.002159, -0.038597,  0.047667, ..., -0.07307 , -0.01004 , -0.002751],
        [-0.002159, -0.038597,  0.047667, ..., -0.07307 , -0.01004 , -0.002751]]], dtype=float32)

In [29]:
x1

array([[[ 0.17385 , -0.048393,  0.226691,  0.53306 , -0.081954, -0.081948],
        [ 0.552724,  0.268602, -0.164316,  0.189896, -0.162196, -0.163005],
        [ 0.084687, -0.669648, -0.603721, -0.196801, -0.354491,  0.109987],
        [-0.317808, -0.494306,  0.512977, -0.079022,  0.023635, -0.498662],
        [-0.190534,  0.038823, -0.402848,  0.131494, -0.210224, -0.102093],
        [-0.210597,  0.648297, -0.004724, -0.370199,  0.287891, -0.427295],
        [ 0.073102, -0.685885, -0.464865,  0.068901,  0.258463,  0.059979],
        [-0.040477, -0.105386, -0.517483, -0.251945, -0.161224,  0.369993],
        [ 0.120266, -0.617064,  0.113429, -0.134779, -0.236923,  0.214087],
        [ 0.36085 ,  0.325948, -0.293726, -0.108224,  0.115942,  0.341441],
        [-0.167711, -0.064981, -0.387217, -0.418672,  0.284384,  0.474684],
        [-0.025204,  0.351236,  0.126573, -0.225792,  0.126488,  0.538313],
        [-0.012539,  0.547625, -0.916911,  0.287666,  0.030466, -0.104653],
        [ 0.